In [1]:
from pathlib import Path
import os
import time
import sys
import json
import gc

repo_dir =  Path.cwd()
os.environ["HF_HOME"] = str(repo_dir / ".hf_cache")
sys.path.insert(0, str(repo_dir / "src"))

from lp_llm.solver import solve_lp_problem
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

<python-env>\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1108 20:31:26.097000 15372 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
print(f"VRAM       : {torch.cuda.memory_allocated() / 1024**3:.2f} Go")
print(f"Reserved VRAM       : {torch.cuda.memory_reserved() / 1024**3:.2f} Go")
a,b = torch.cuda.mem_get_info()
print(f"Free VRAM  : {a/1024**3:.2f} Go")
print(f"Total VRAM : {b/1024**3:.2f} Go")

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
t0 = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(model_id,revision="989aa7980e4cf806f80c7fef2b1adb7bc71aa306")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
    revision="989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
)

print(model.dtype)  # torch.bfloat16
model.eval()

torch.cuda.synchronize()
loading_time = time.perf_counter() - t0

n_params = sum(p.numel() for p in model.parameters())
print(f"Paramètres : {n_params:,} ({n_params / 1e9:.2f} B)")

# Dtype et device
p = next(model.parameters())
print(f"Dtype      : {p.dtype}")
print(f"Device     : {p.device}")

# Configuration du modèle
config = model.config
print(f"Couches    : {config.num_hidden_layers}")
print(f"Hidden size: {config.hidden_size}")
print(f"Contexte max : {config.max_position_embeddings}")

# Tokenizer
print(f"Vocabulaire : {len(tokenizer)}")
print(f"EOS token   : {tokenizer.eos_token} ({tokenizer.eos_token_id})")
print(f"PAD token   : {tokenizer.pad_token} ({tokenizer.pad_token_id})")

VRAM       : 0.00 Go
Reserved VRAM       : 0.00 Go
Free VRAM  : 6.94 Go
Total VRAM : 8.00 Go


Loading weights: 100%|██████████| 338/338 [00:02<00:00, 132.30it/s]

torch.bfloat16
Paramètres : 1,543,714,304 (1.54 B)
Dtype      : torch.bfloat16
Device     : cuda:0
Couches    : 28
Hidden size: 1536
Contexte max : 32768
Vocabulaire : 151665
EOS token   : <|im_end|> (151645)
PAD token   : <|endoftext|> (151643)


In [3]:
messages = [ 
    {"role": "system", 
    "content": """
    You are producing a json, no more, no less. 
    It should represent a linear problem in the form of for example : json is {
    "variables": ["x1", "x2"],
    "objective": {"sense": "min", "coefficients": [5, 7.5]},
    "constraints": [
        {"coefficients": [1, 1], "sense": "<=", "rhs": 3},
        {"coefficients": [1, 1], "sense": "<=", "rhs": 0.5}
    ],
    "bounds": [[0, null], [0.1, 12]]
}.
Constraints on the way to write the problem : 
 1 - Always 2 variables, named x1 and x2. 
 2 - json["objective"]["coefficients"] must be a list of length 2 (because 2 variables).
 3 -  json["objective"]["sense"] is in {"min","max"}
 4 - json["constraints"] must be a list of length 2 or 3
 5 - for all i, json["constraints"][i]["coefficients"] must be a list of length 2 (because 2 variables).
 6 - for all i, json["constraints"][i]["sense"] is in {"<=","=",">="}
 7 - json["bounds"] must be a list of length 2 (because 2 variables).
 8 - for all i, json["bounds"][i] must be a list of length 2
 9 - for all i, json["bounds"][i][0] can't be negative
 10 - for all i, json["bounds"][i][0]<= json["bounds"][i][1]
 11 - "coefficients", "rhs" and elements of bounds should be numbers ( int or float)
 12 - for json["bounds"][i][j], null represent no bound, ie +infinity
 13 - sometimes "coefficients" and "rhs" can be presented to you with different units, e.g. hour and minutes, convert, and put results directly in the json.
 
 forbidden characters : ``` or //
 Answer only with the valid JSON (all your generated text could pass a json.loads), without text or markdown .
"""}, 
{"role": "user", "content": """
A bakery produces two types of products: **cakes and cookies**.

The bakery has a daily supply of **20 kg of flour** and a maximum of **8 working hours** available each day.

Each product requires different resources and generates a different profit:

| | Cakes | Cookies |
|---|---|---|
| Flour required per unit | 500 g | 200 g |
| Preparation time per unit | 30 minutes | 10 minutes |
| Profit per unit | €12 | €4 |

**Objective:** Maximize the bakery's total daily profit.

**Decision variables:**
- The number of cakes produced each day.
- The number of cookies produced each day.

**Constraints:**
1. **Flour availability:** The total amount of flour used cannot exceed 20 kg per day.
2. **Working time:** The total preparation time cannot exceed 8 hours per day.

**Question:** How many cakes and cookies should the bakery produce each day to maximize its profit while respecting both constraints?
"""
}]

lp_json = """
{
  "variables": ["x1", "x2"],
  "objective": {"sense": "max", "coefficients": [12, 4]},
  "constraints": [
    {"coefficients": [0.5, 0.2], "sense": "<=", "rhs": 20},
    {"coefficients": [30, 10], "sense": "<=", "rhs": 480} 
  ],
  "bounds": [[0, null], [0, null]]
}
"""

target = json.dumps(json.loads(lp_json))

assert solve_lp_problem(target)["status"] == "optimal"

full_msgs = messages + [{"role": "assistant", "content": target}]

prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
full_text = tokenizer.apply_chat_template(full_msgs, tokenize=False)

assert full_text.startswith(prompt_text)  # le début doit être identique

prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]

assert full_ids[:len(prompt_ids)] == prompt_ids  # la frontière ne doit pas couper un token

labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
if labels[-1]==198:
    labels = labels[:-1]
    full_ids = full_ids[:-1]

assert len(labels) == len(full_ids)
trained = [t for t, l in zip(full_ids, labels) if l != -100]
print(repr(tokenizer.decode(trained)))
print(f"length of prompt: {len(prompt_ids)}")
print(f"length of full  : {len(full_ids)}")
print(f"length of target: {len(full_ids) -len(prompt_ids) }")

'{"variables": ["x1", "x2"], "objective": {"sense": "max", "coefficients": [12, 4]}, "constraints": [{"coefficients": [0.5, 0.2], "sense": "<=", "rhs": 20}, {"coefficients": [30, 10], "sense": "<=", "rhs": 480}], "bounds": [[0, null], [0, null]]}<|im_end|>'
length of prompt: 705
length of full  : 805
length of target: 100


In [4]:
sorted({n.split(".")[-1] for n, _ in model.named_modules() if n.endswith(("proj"))})

['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj']

In [5]:
torch.manual_seed(42)
config = LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,target_modules=["q_proj", "k_proj", "v_proj", "o_proj" ],bias = "none", task_type="CAUSAL_LM")
peft_model = get_peft_model(model, config)
peft_model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [6]:
optimizer = torch.optim.AdamW([p for p in peft_model.parameters() if p.requires_grad], lr=1e-4)

In [ ]:
def run(n_steps):
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    losses = []
    times = []
    for step in range(n_steps):
        t0 = time.perf_counter()
        optimizer.zero_grad()
        out = peft_model(input_ids=input_ids, labels=labels_ids)
        out.loss.backward()
        optimizer.step()
        torch.cuda.synchronize()
        loading_time = time.perf_counter() - t0
        losses.append(out.loss.item())
        times.append(loading_time)
        print(f"step {step} : loss {out.loss.item():.4f}")

    torch.cuda.synchronize()
    
    del out
    return losses,sum(times)/len(times),torch.cuda.max_memory_allocated() 

In [ ]:
peft_model.train()
input_ids = torch.tensor([full_ids], device=peft_model.device)
labels_ids = torch.tensor([labels], device=peft_model.device)
#out = peft_model(input_ids=input_ids, labels=labels_ids)

init_lora = {k: v.detach().clone() for k, v in peft_model.state_dict().items() if "lora_" in k} #save for memory comp with peft_model.gradient_checkpointing_enable

losses,loading_time,max_memory_allocated = run(5)

print(f"VRAM pic : {max_memory_allocated / 1024**3:.2f} Go")
print(f"tps 5 boucles : {loading_time:.2f} s")


step 0 : loss 0.1837
step 1 : loss 0.1541
step 2 : loss 0.1090
step 3 : loss 0.0672
step 4 : loss 0.0365
VRAM pic : 7.84 Go
tps 5 boucles : 9.95 s


In [ ]:

peft_model.load_state_dict(init_lora, strict=False)   # adaptateurs remis à l'état initial
del optimizer
gc.collect()
torch.cuda.empty_cache()                               # rend la mémoire réservée au pilote
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

optimizer = torch.optim.AdamW([p for p in peft_model.parameters() if p.requires_grad], lr=1e-4)
peft_model.gradient_checkpointing_enable()
torch.manual_seed(42)

peft_model.train()
input_ids = torch.tensor([full_ids], device=peft_model.device)
labels_ids = torch.tensor([labels], device=peft_model.device)

losses,loading_time,max_memory_allocated = run(5)

print(f"VRAM pic : {torch.cuda.max_memory_allocated() / 1024**3:.2f} Go")
print(f"tps 5 boucles : {loading_time:.2f} s")

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


step 0 : loss 0.1837
step 1 : loss 0.1541
step 2 : loss 0.1090
step 3 : loss 0.0672
step 4 : loss 0.0365
VRAM pic : 4.62 Go
tps 5 boucles : 3.11 s
